## Imports

In [ ]:
from pathlib import Path
import json
import numpy as np

from mobile_motion_planning.ik_offline.geometry import Plane
from mobile_motion_planning.ik_offline.trans import from_plane_to_plane, apply_T_to_plane

tcp_in_world = Plane((0.000,-0.29786,0.361), (0.0151,0,0), (0,-0.014,0.0056)) # tool 8 abele split 90deg adapter straight inlet

# JSON_PATH = Path('../data/auto_generated/export/251117_163017_planes.json')
JSON_PATH = Path('../data/example_data/__251117_163017_flange_frames.json')
print(f"Using JSON file: {JSON_PATH.resolve()}")

def load_planes(json_path: Path):
    with json_path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    planes = []
    for p in data:
        planes.append(Plane(origin=tuple(p['origin']), xaxis=tuple(p['x_axis']), yaxis=tuple(p['y_axis']), zaxis=tuple(p['z_axis'])))
    return planes


def flange_from_plane(target_plane: Plane) -> Plane:
    """Compute flange frame in world for a target TCP plane (no IK)."""
    T = from_plane_to_plane(tcp_in_world, target_plane)
    flange = apply_T_to_plane(T)
    return flange

def flange_for_all(planes):
    return [flange_from_plane(p) for p in planes]
    
def plane_to_dict(plane: Plane):
    return {
        "origin": np.asarray(plane.origin, dtype=float).tolist(),
        "x_axis": np.asarray(plane.xaxis, dtype=float).tolist(),
        "y_axis": np.asarray(plane.yaxis, dtype=float).tolist(),
        "z_axis": np.asarray(plane.zaxis, dtype=float).tolist() if plane.zaxis is not None else [],
    }

## From Flange_Frames to Base_Frames

In [ ]:
flange_planes = load_planes(JSON_PATH)
print(f"Loaded {len(planes)} planes")

# create base planes for testing by projecting the origin of each flange plane down to Z=0 and aligning axes with world XY
base_planes = []
x_offset = 0.7
y_offset = 0
z_offset = 1.1
for flange in flange_planes:
    base_origin = (flange.origin[0] + x_offset, flange.origin[1] + y_offset, z_offset)
    base_xaxis = (1, 0, 0)
    base_yaxis = (0, 1, 0)
    base_zaxis = (0, 0, 1)
    base_plane = Plane(origin=base_origin, xaxis=base_xaxis, yaxis=base_yaxis, zaxis=base_zaxis)
    base_planes.append(base_plane)

print(f"Created {len(base_planes)} base planes for testing")
#save
out_base_path = Path('../data/example_data/___251117_163017_base_frames.json')

out_base_path.write_text(
    json.dumps([plane_to_dict(p) for p in base_planes], indent=2),
    encoding="utf-8",
)
print(f"Saved base frames to {out_base_path}")

Using JSON file: C:\Users\david\Documents\GitHub\mobile_motion_planning\mobile_motion_planning\data\example_data\__251117_163017_flange_frames.json


## From Target_Frames to Flange_Frames to Base_Frames

In [ ]:
planes = load_planes(JSON_PATH)
print(f"Loaded {len(planes)} planes")
# TCP in world as defined in ik_offline.py
# tcp_in_world = Plane((0.00302, -0.29571, 0.40445), (0.0366, 0, 0), (0, 0.0051, 0.0141)) # tool 7 abele split 90deg adapter straight inlet

flange_planes = flange_for_all(planes)
print(f"Computed {len(flange_planes)} flange frames")

# save_flange_frames
out_path = Path("../data/example_data/___251117_163017_flange_frames.json")
out_path.write_text(
    json.dumps([plane_to_dict(p) for p in flange_planes], indent=2),
    encoding="utf-8",
)
print(f"Saved flange frames to {out_path}")


# create base planes
base_planes = []
x_offset = 0.7
y_offset = 0
z_offset = 1.1
for flange in flange_planes:
    base_origin = (flange.origin[0] + x_offset, flange.origin[1] + y_offset, z_offset)
    base_xaxis = (1, 0, 0)
    base_yaxis = (0, 1, 0)
    base_zaxis = (0, 0, 1)
    base_plane = Plane(
        origin=base_origin, xaxis=base_xaxis, yaxis=base_yaxis, zaxis=base_zaxis
    )
    base_planes.append(base_plane)

print(f"Created {len(base_planes)} base planes")

# save
out_base_path = Path("../data/example_data/___251117_163017_base_frames.json")
out_base_path.write_text(
    json.dumps([plane_to_dict(p) for p in base_planes], indent=2),
    encoding="utf-8",
)
print(f"Saved base frames to {out_base_path}")

Computed 1910 flange frames


[Plane(origin=array([0.7965494685651201, 0.0288708003675598, 2.84764623975236  ]), xaxis=array([ 0.7025211723906232, -0.5032216223210783,  0.5032216223210668]), yaxis=array([-0.515342058844238 , -0.847380786453314 , -0.1279389117350354]), zaxis=array([ 0.4908019608040359, -0.1694514726353986, -0.8546342104623526])),
 Plane(origin=array([0.8082535539526883, 0.0165546251449757, 2.859962414974944 ]), xaxis=array([ 0.7656776923840748, -0.4548283584965914,  0.4548283584965809]), yaxis=array([-0.4657832102032087, -0.8797196304130999, -0.0956000677752502]), zaxis=array([ 0.443603057336389 , -0.1386525736260786, -0.8854331094716716]))]

In [25]:
angles_rad=[2.8926299262448385,
      -0.020992072403053008,
      -1.935998521206888,
      0.003168722006412242,
      1.75670556601428,
      0.007152186822537487]
angles_deg = np.degrees(angles_rad)
print(angles_deg)

[ 165.73548646706786     -1.2027571519279854 -110.92454440872328
    0.1815543974176476  100.65181477975868      0.4097901192204807]


In [26]:
from mobile_motion_planning.ik_offline.ik import forward_kinematics

print(forward_kinematics(angles_deg.tolist()))


Plane(origin=array([ 0.6660590240319464, -0.1516405068223914,  0.3576355346646182]), xaxis=array([-0.0720780362944453, -0.0841206300438699, -0.993845297963903 ]), yaxis=array([-0.6843779776278907,  0.7290274701818051, -0.0120719285234628]), zaxis=array([ 0.7255560215599447,  0.6792957141932662, -0.1101171751216195]))
